# Dermatological multimodal dataset · Part 1: Exploratory Data Analysis
**Author:** Marco Larriba Rius

In [ ]:
# == Core libraries ============================================================
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.ticker as ticker
import seaborn as sns, warnings, os, glob
from pathlib import Path
from scipy.stats import chi2_contingency, spearmanr
from sklearn.model_selection import train_test_split, GroupShuffleSplit
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=0.9)
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlepad':6,'axes.labelpad':4})
RANDOM_STATE=42
TARGET='diagnostic'
MALIGNANT={'BCC','MEL','SCC','BOD'}   # BOD = Bowen's = SCC in situ

# Locate the local _data folder robustly (works regardless of the kernel's working directory).
def _locate_data(name='_data'):
    p = Path.cwd()
    for base in [p, *p.parents][:6]:
        if (base/name/'images').is_dir() or (base/name/'metadata_skin_cancer.csv').exists():
            return base/name
    for hit in glob.glob(str(p/'**'/name/'metadata_skin_cancer.csv'), recursive=True):
        return Path(hit).parent
    return Path(name)
DATA_DIR    = _locate_data()
META_PATH   = str(DATA_DIR/'metadata_skin_cancer.csv')
IMG_DIR     = str(DATA_DIR/'images')
FIGURES_DIR = DATA_DIR.parent/'figures'; FIGURES_DIR.mkdir(exist_ok=True)

# Images are loaded LOCALLY from _data/images (unzip archive.zip there). Never Colab/kagglehub.
img_files  = glob.glob(os.path.join(IMG_DIR, '**', '*.png'), recursive=True)
RUN_IMAGES = len(img_files) > 0
print(f'DATA_DIR = {DATA_DIR}')
print(f'Local images: {len(img_files)} PNGs  ->  RUN_IMAGES = {RUN_IMAGES}')
if not RUN_IMAGES:
    print('  (unzip archive.zip into _data/images to enable the image cells)')

## Step 1 - Load and first look

In [ ]:
df=pd.read_csv(META_PATH)
print('Shape:', df.shape)
df.head(3)

## Step 2 - Exploratory Data Analysis: tabular data
Discovery only — **no variable is modified here**; every transformation is deferred to the preprocessing proposals (Step 5).

### 2.1 Statistical audit

In [ ]:
# Per-column audit: type, distinct values, missing, sample values and quality flags.
# ID/HIGH-CARD = near-unique identifier; BOOL-LIKE = boolean-ish (incl. 'UNK'); CONSTANT / CATEGORICAL-LOW.
def audit(data):
    n = len(data); rows = []
    BOOLSET = {'TRUE','FALSE','YES','NO','UNK','1','0','1.0','0.0'}
    for c in data.columns:
        s = data[c]; nun = s.nunique(dropna=True); miss = s.isna().mean()*100; ratio = nun/n
        vals = {str(v).strip().upper() for v in s.dropna().unique()}
        flags = []
        if nun <= 1: flags.append('CONSTANT')
        if ratio > 0.5: flags.append('ID/HIGH-CARD')
        if vals and vals <= BOOLSET: flags.append('BOOL-LIKE')
        elif s.dtype == object and 0 < nun <= 4: flags.append('CATEGORICAL-LOW')
        sample = ', '.join(map(str, s.dropna().unique()[:3]))[:38]
        rows.append({'column': c, 'dtype': str(s.dtype), 'n_unique': nun,
                     'n_missing': int(s.isna().sum()), 'pct_missing': round(miss,1),
                     'sample_values': sample, 'flags': ';'.join(flags)})
    return pd.DataFrame(rows)
audit_df = audit(df)
print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} columns')
audit_df

In [ ]:
# Statistical audit of the numerical attributes: range, mean, variance.
num_cols = ['age', 'diameter_1', 'diameter_2', 'fitspatrick']
desc = df[num_cols].describe().T
desc['range']    = desc['max'] - desc['min']
desc['variance'] = df[num_cols].var()
desc['skew']     = df[num_cols].skew()
cols = ['count','mean','std','variance','min','25%','50%','75%','max','range','skew']
print('Numerical statistics (range / mean / variance / quartiles / skew):')
print(desc[cols].round(2).to_string())

In [ ]:
# Outlier detection via IQR (Tukey) fences + boxplots — observation only.
rep = {}
for c in num_cols:
    s = df[c].dropna(); q1, q3 = s.quantile(.25), s.quantile(.75); iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr; nout = int(((s < lo) | (s > hi)).sum())
    rep[c] = {'lower': round(lo,1), 'upper': round(hi,1), 'n_outliers': nout, 'pct': round(nout/len(s)*100,1)}
print('IQR outlier report:'); print(pd.DataFrame(rep).T.to_string())
fig, axes = plt.subplots(1, 4, figsize=(13, 3.0))
for ax, c in zip(axes, num_cols):
    sns.boxplot(y=df[c].dropna(), ax=ax, color='#5DADE2'); ax.set_title(c, fontsize=9, fontweight='bold'); ax.set_ylabel('')
fig.suptitle('Numerical attributes — IQR boxplots', fontsize=10, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_boxplots.png', bbox_inches='tight'); plt.show()

In [ ]:
# Hierarchical multiplicity (why a naive split would leak): images per patient / lesions per patient.
print('Patients: %d | lesions: %d | images(rows): %d' % (df.patient_id.nunique(), df.lesion_id.nunique(), len(df)))
ipp = df.groupby('patient_id').size(); lpp = df.groupby('patient_id')['lesion_id'].nunique()
print('Images per patient : mean=%.2f max=%d' % (ipp.mean(), ipp.max()))
print('Lesions per patient: mean=%.2f max=%d' % (lpp.mean(), lpp.max()))
fig, ax = plt.subplots(figsize=(6, 3.0))
ipp.value_counts().sort_index().plot.bar(ax=ax, color='#AED6F1', edgecolor='#2471A3')
ax.set_xlabel('images per patient'); ax.set_ylabel('n patients')
ax.set_title('Images per patient (hierarchical multiplicity)', fontsize=9.5, fontweight='bold')
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_multiplicity.png', bbox_inches='tight'); plt.show()

In [ ]:
# Categorical / boolean values AS-IS: surface the inconsistent encodings (anomalies to fix LATER, Step 5).
bool_like = ['smoke','drink','pesticide','skin_cancer_history','cancer_history','has_piped_water',
             'has_sewage_system','itch','grew','hurt','changed','bleed','elevation','biopsed']
print('Distinct raw values per boolean-like column (note mixed True/False vs TRUE/FALSE and UNK):')
for c in bool_like:
    print(f'  {c:20s}: {sorted(map(str, df[c].dropna().unique()))}')
unk = {c: int((df[c].astype(str).str.upper()=="UNK").sum()) for c in ['itch','grew','hurt','changed','bleed','elevation']}
print('\nUNK ("not assessed") counts in symptoms (informative missingness):', unk)

### 2.2 Missing-value structure — is it random? (MNAR test)

In [ ]:
block=['smoke','drink','pesticide','gender','skin_cancer_history','cancer_history',
       'has_piped_water','has_sewage_system','fitspatrick','diameter_1','diameter_2',
       'background_father','background_mother']
miss=(df[block].isna().mean()*100).round(1)
nullind=df[block].isna().astype(int)
# how often the whole block is missing together
all_missing=(nullind.sum(axis=1)==len(block)).mean()*100
none_missing=(nullind.sum(axis=1)==0).mean()*100
print(f'Rows with the ENTIRE clinical block missing: {all_missing:.1f}%')
print(f'Rows with NONE of the block missing:        {none_missing:.1f}%')
# does missingness align with biopsy status / hospital cohort?
df['_block_missing']=(nullind.sum(axis=1)>=len(block)-1).astype(int)
print('\nBlock-missing rate by biopsed:'); print(df.groupby('biopsed')['_block_missing'].mean().round(3).to_string())
ct=pd.crosstab(df['hospital_id'],df['_block_missing'],normalize='index')[1].round(2).sort_values()
print('\nBlock-missing rate by hospital (head/tail):'); print(ct.head(4).to_string()); print('...'); print(ct.tail(4).to_string())

# Formal MNAR test: is block-missingness independent of biopsy / outcome? (MCAR would say yes)
for var in ['biopsed', 'diagnostic']:
    chi, p, dof, _ = chi2_contingency(pd.crosstab(df['_block_missing'], df[var]))
    print(f'  block-missing vs {var:11s}: chi2={chi:8.1f}  dof={dof}  p={p:.2e}')

# Two clear panels: (A) every clinical variable shares ~the same missing rate (they go missing
# TOGETHER), and (B) that block-missingness is driven by biopsy status -> structural, not random.
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
miss.sort_values().plot.barh(ax=axes[0], color='#5DADE2', edgecolor='white')
axes[0].set_xlabel('% missing'); axes[0].tick_params(labelsize=7.5)
axes[0].set_title('(A) Missing rate per clinical variable', fontsize=9.5, fontweight='bold')
rate = df.groupby('biopsed')['_block_missing'].mean()*100
vals = [rate.get(False, 0), rate.get(True, 0)]
bars = axes[1].bar(['not biopsied', 'biopsied'], vals, color=['#E74C3C', '#58D68D'], edgecolor='white')
for bar, v in zip(bars, vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)
axes[1].set_ylim(0, 100); axes[1].set_ylabel('% with the whole clinical block missing')
axes[1].set_title('(B) Block-missing rate by biopsy status', fontsize=9.5, fontweight='bold')
fig.suptitle('Clinical-block missingness is structural (MNAR), driven by biopsy status',
             fontsize=10, fontweight='bold', y=1.04)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig1_missingness.png', bbox_inches='tight'); plt.show()
df.drop(columns='_block_missing', inplace=True)

### 2.3 Correlation analysis

In [ ]:
# Numerical correlation (Spearman) -> redundant features (shown as a table, not a heatmap)
num_cols = ['age', 'diameter_1', 'diameter_2', 'fitspatrick']
print('Spearman correlation between numerical attributes:')
print(df[num_cols].corr(method='spearman').round(2).to_string())
r, p = spearmanr(df['diameter_1'], df['diameter_2'], nan_policy='omit')
print(f'\ndiameter_1 vs diameter_2: r={r:.3f} (p={p:.1e}) -> redundant; keep one or derive area.')

In [ ]:
def cramers_v(a, b):
    """Bias-corrected Cramér's V (Bergsma) with its chi-square p-value (0..1; p<0.05 = significant)."""
    t = pd.crosstab(a, b); n = t.to_numpy().sum()
    if n == 0 or t.shape[0] < 2 or t.shape[1] < 2: return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(t); phi2 = chi2 / n; r, k = t.shape
    phi2c = max(0, phi2 - (k-1)*(r-1)/(n-1))
    rc = r - (r-1)**2/(n-1); kc = k - (k-1)**2/(n-1); denom = min(kc-1, rc-1)
    return (np.sqrt(phi2c/denom) if denom > 0 else np.nan), p

cat_cols=['gender','region','smoke','drink','pesticide','skin_cancer_history','cancer_history',
          'itch','grew','hurt','changed','bleed','elevation','biopsed',TARGET]
M=pd.DataFrame(index=cat_cols,columns=cat_cols,dtype=float)
for i in cat_cols:
    for j in cat_cols:
        M.loc[i,j]=1.0 if i==j else cramers_v(df[i].astype(str),df[j].astype(str))[0]
fig,ax=plt.subplots(figsize=(8.5,7))
sns.heatmap(M.astype(float),annot=True,fmt='.2f',cmap='viridis',square=True,ax=ax,annot_kws={'size':6.5},
            cbar_kws={'shrink':0.7,'label':"Cramér's V"})
ax.set_title("Categorical association (Cramér's V) with diagnostic", fontsize=10, fontweight='bold')
ax.tick_params(labelsize=7)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig4_cramers.png', bbox_inches='tight'); plt.show()
print('Association with diagnostic (Cramér\'s V, chi-square p):')
for c in [x for x in cat_cols if x!=TARGET]:
    v,p=cramers_v(df[c].astype(str),df[TARGET].astype(str)); print(f'  {c:20s} V={v:.3f}  p={p:.2e}')
# Relationships BETWEEN variables (not only with the target) -> relationships between attributes.
Mv = M.drop(index=TARGET, columns=TARGET)
pairs = Mv.where(np.triu(np.ones(Mv.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("\nStrongest associations BETWEEN variables (Cramer's V):")
print(pairs.head(8).round(3).to_string())

In [ ]:
# Are the strong associations real, or an artefact of the MNAR clinical block?
# The block only exists on biopsied lesions; re-estimate V on biopsied-only rows.
biop = df[df['biopsed'].astype(str).str.upper() == 'TRUE']
suspect = ['smoke','drink','pesticide','gender','skin_cancer_history','cancer_history']
symptom = ['grew','changed','itch','elevation','bleed','region']
rows = []
for c in suspect + symptom:
    v_all, _  = cramers_v(df[c].astype(str),  df[TARGET].astype(str))
    v_b,  p_b = cramers_v(biop[c].astype(str), biop[TARGET].astype(str))
    rows.append((c, round(v_all,3), round(v_b,3), f'{p_b:.1e}',
                 'lifestyle/admin' if c in suspect else 'symptom/lesion'))
spur = pd.DataFrame(rows, columns=['feature','V_full','V_biopsied','p_biopsied','type'])
print(spur.to_string(index=False))
print('\nLifestyle/admin associations collapse on biopsied-only rows (artefact of the shared MNAR'
      ' block); symptom/lesion signals persist -> the latter are the real clinical signal.')

### 2.4 Visual inspection

In [ ]:
# Target class distribution (the key imbalance) + malignant vs benign grouping
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
vc = df[TARGET].value_counts()
axes[0].bar(vc.index, vc.values, color='#5DADE2', edgecolor='white')
for i, v in enumerate(vc.values):
    axes[0].text(i, v+8, f'{v}\n{v/len(df)*100:.1f}%', ha='center', fontsize=8, fontweight='bold')
axes[0].set_title('diagnostic (6 classes)', fontsize=9.5, fontweight='bold'); axes[0].set_ylim(0, vc.max()*1.2)
df['is_malignant'] = df[TARGET].isin(MALIGNANT).astype(int)
mb = df['is_malignant'].value_counts()
axes[1].bar(['Benign', 'Malignant'], [mb.get(0,0), mb.get(1,0)], color=['#58D68D', '#E74C3C'], edgecolor='white')
for i, v in enumerate([mb.get(0,0), mb.get(1,0)]):
    axes[1].text(i, v+8, f'{v/len(df)*100:.1f}%', ha='center', fontsize=8, fontweight='bold')
axes[1].set_title('malignant vs benign', fontsize=9.5, fontweight='bold')
fig.suptitle('Target class distribution (severe imbalance)', fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig2_distributions.png', bbox_inches='tight'); plt.show()
print('Class imbalance: BCC %.1f%% vs MEL %.1f%% -> ratio %.1fx'
      % (vc.max()/len(df)*100, vc.min()/len(df)*100, vc.max()/vc.min()))

In [ ]:
# Numeric distributions
num_plot=['age','diameter_1','fitspatrick']
fig,axes=plt.subplots(1,3,figsize=(12,3.2)); axes=axes.flatten()
for ax,col in zip(axes,num_plot):
    data=df[col].dropna(); ax.hist(data,bins=20,color='#5DADE2',edgecolor='white',alpha=0.9)
    ax.axvline(data.mean(),color='#c0392b',lw=1.3,ls='--'); ax.axvline(data.median(),color='#e67e22',lw=1.3,ls=':')
    ax.set_title(f'{col}  (skew={data.skew():.2f})', fontsize=9, fontweight='bold'); ax.tick_params(labelsize=7)
fig.suptitle('Numerical distributions  (-- mean  ·· median)', fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig3_numeric.png', bbox_inches='tight'); plt.show()

In [ ]:
# Count plots for the KEY categorical / boolean variables (raw values, NaN shown).
# Full distribution of every categorical column is in the summary table below.
key_cat = ['region', 'biopsed', 'grew']
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6)); axes = axes.ravel()
for ax, c in zip(axes, key_cat):
    s = df[c].astype(str).where(df[c].notna(), 'NA')
    vc = s.value_counts().head(8)
    ax.bar(range(len(vc)), vc.values, color='#5DADE2', edgecolor='white')
    ax.set_xticks(range(len(vc))); ax.set_xticklabels(vc.index, rotation=30, ha='right', fontsize=7)
    ax.set_title(c, fontsize=9, fontweight='bold'); ax.tick_params(axis='y', labelsize=7)
    for j, v in enumerate(vc.values):
        ax.text(j, v, f'{v/len(df)*100:.0f}%', ha='center', va='bottom', fontsize=6.5)
fig.suptitle('Distribution of key categorical and boolean variables', fontsize=10, fontweight='bold', y=1.0)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_countplots.png', bbox_inches='tight'); plt.show()

In [ ]:
# Categorical & boolean distributions as a compact TABLE (clearer in the report than many subplots).
def dist_summary(col, top=3):
    s = df[col]; n = len(s)
    vc = s.astype(str).where(s.notna(), 'NA').value_counts()
    parts = [f'{k} {v/n*100:.0f}%' for k, v in vc.head(top).items()]
    return {'n_unique': int(s.nunique(dropna=True)), 'missing_%': round(s.isna().mean()*100, 1),
            'top_values': ', '.join(parts)}
cat_all = ['gender','region','province','city','hospital_id','smoke','drink','pesticide',
           'skin_cancer_history','cancer_history','has_piped_water','has_sewage_system',
           'itch','grew','hurt','changed','bleed','elevation','biopsed']
cat_table = pd.DataFrame({c: dist_summary(c) for c in cat_all}).T
print('Categorical / boolean distribution summary (value shares; NA = missing):')
print(cat_table.to_string())

In [ ]:
# Class overlap of numeric attributes (benign vs malignant): KDE shows distribution overlap.
num_plot = ['age','diameter_1','fitspatrick']
fig,axes=plt.subplots(1,3,figsize=(12,3.2)); axes=axes.flatten()
for ax,col in zip(axes,num_plot):
    for lab,color in [(0,'#58D68D'),(1,'#E74C3C')]:
        sns.kdeplot(df[df['is_malignant']==lab][col].dropna(), ax=ax, fill=True, alpha=0.35,
                    color=color, label=('Malignant' if lab else 'Benign'), warn_singular=False)
    med0=df[df.is_malignant==0][col].median(); med1=df[df.is_malignant==1][col].median()
    ax.set_title(f'{col}  (median B={med0:.0f}, M={med1:.0f})', fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7); ax.legend(fontsize=6, frameon=False)
fig.suptitle('Class overlap of numerical attributes (benign vs malignant)', fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig8_overlap.png', bbox_inches='tight'); plt.show()

## Step 3 - Exploratory Data Analysis: image data
*Runs locally with the images unzipped in `_data/images`. Each row's `img_id` maps to a PNG file.*

In [ ]:
if RUN_IMAGES:
    from PIL import Image
    from scipy.stats import entropy as img_entropy
    files = {os.path.basename(p): p for p in glob.glob(os.path.join(IMG_DIR, '**', '*.png'), recursive=True)}
    sample = df   # audit ALL images (not a subsample)
    rec = []
    for _, r in sample.iterrows():
        p = files.get(str(r['img_id']))
        if not p: continue
        im = Image.open(p); arr = np.asarray(im.convert('RGB')); gray = np.asarray(im.convert('L'))
        hist = np.bincount(gray.ravel(), minlength=256) + 1
        rec.append({'diagnostic': r['diagnostic'], 'w': im.size[0], 'h': im.size[1],
                    'aspect': round(im.size[0]/im.size[1], 3), 'mode': im.mode,
                    'brightness': arr.mean(), 'contrast': arr.std(),
                    'R': arr[..., 0].mean(), 'G': arr[..., 1].mean(), 'B': arr[..., 2].mean(),
                    'entropy': float(img_entropy(hist, base=2))})
    idf = pd.DataFrame(rec)
    print(f'Technical audit on ALL {len(idf)} images:')
    print(idf[['w','h','aspect','brightness','contrast','entropy']].describe().round(1).to_string())
    print('\nColour modes:', idf['mode'].value_counts().to_dict())
    print('Unique (w,h) resolutions in sample: %d / %d  -> highly heterogeneous acquisition'
          % (idf.groupby(['w','h']).ngroups, len(idf)))
else:
    print('SKIPPED (no local images). Unzip archive.zip into _data/images.')

In [ ]:
if RUN_IMAGES:
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    ax[0].scatter(idf['w'], idf['h'], s=8, alpha=0.4, color='#2471A3')
    ax[0].set_xlabel('width (px)'); ax[0].set_ylabel('height (px)')
    ax[0].set_title('Resolution spread', fontsize=9.5, fontweight='bold')
    ax[1].hist(idf['aspect'], bins=30, color='#5DADE2', edgecolor='white')
    ax[1].set_xlabel('aspect ratio (w/h)'); ax[1].set_title('Aspect ratio', fontsize=9.5, fontweight='bold')
    idf['mode'].value_counts().plot.bar(ax=ax[2], color='#AED6F1', edgecolor='#2471A3')
    ax[2].tick_params(axis='x', rotation=0); ax[2].set_title('Colour mode', fontsize=9.5, fontweight='bold')
    fig.suptitle('Image acquisition consistency (resolution, aspect ratio, colour mode)',
                 fontsize=10, fontweight='bold', y=1.03)
    plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_img_tech.png', bbox_inches='tight'); plt.show()
else:
    print('SKIPPED')

In [ ]:
if RUN_IMAGES:
    feats = ['brightness', 'contrast', 'entropy']
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    for a, f in zip(ax, feats):
        sns.boxplot(data=idf, x='diagnostic', y=f, ax=a, color='#5DADE2')
        a.set_title(f, fontsize=9, fontweight='bold'); a.set_xlabel(''); a.tick_params(axis='x', rotation=30, labelsize=7)
    fig.suptitle('Image features across diagnostic classes', fontsize=10, fontweight='bold', y=1.03)
    plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig5_image_stats.png', bbox_inches='tight'); plt.show()
    print('Compare the per-class medians/boxes: if they overlap heavily, global image statistics do '
          'NOT separate the classes (the diagnostic signal is local to the lesion, not overall brightness).')
else:
    print('SKIPPED')

In [ ]:
if RUN_IMAGES:
    from PIL import Image
    classes = sorted(df[TARGET].unique()); SZ = (128, 128); PER = 40
    fig, axes = plt.subplots(1, len(classes), figsize=(2.2*len(classes), 2.7))
    for ax, cls in zip(axes, classes):
        acc = np.zeros((*SZ, 3)); n = 0
        for iid in df[df[TARGET] == cls]['img_id'].astype(str):
            p = files.get(iid)
            if not p: continue
            acc += np.asarray(Image.open(p).convert('RGB').resize(SZ)); n += 1
            if n >= PER: break
        if n: ax.imshow((acc/n).astype('uint8'))
        ax.set_title(f'{cls} (n={n})', fontsize=8, fontweight='bold'); ax.axis('off')
    fig.suptitle('Mean image per diagnostic class (systematic colour / framing patterns & artefacts)',
                 fontsize=10, fontweight='bold', y=1.06)
    plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_meanimg.png', bbox_inches='tight'); plt.show()
else:
    print('SKIPPED')

In [ ]:
if RUN_IMAGES:
    from PIL import Image
    classes = sorted(df[TARGET].unique())
    fig, axes = plt.subplots(2, 3, figsize=(10, 6.5)); axes = axes.flatten()
    for ax, cls in zip(axes, classes):
        row = df[df[TARGET] == cls].iloc[0]; p = files.get(str(row['img_id']))
        if p: ax.imshow(Image.open(p)); ax.set_title(cls, fontsize=10, fontweight='bold')
        ax.axis('off')
    for k in range(len(classes), len(axes)): axes[k].axis('off')
    fig.suptitle('Representative lesion per diagnostic class', fontsize=10, fontweight='bold')
    plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig6_samples.png', bbox_inches='tight'); plt.show()
else:
    print('SKIPPED')

## Step 4 - Attribute analysis

In [ ]:
# Relationships demographic/clinical vs diagnostic
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
# Left: benign vs malignant share per age band (stacked, legend placed BELOW so it never covers bars)
ageb = pd.cut(df['age'], [0, 30, 50, 70, 120], labels=['<=30', '31-50', '51-70', '>70'])
ct = pd.crosstab(ageb, df['is_malignant'], normalize='index').rename(columns={0: 'Benign', 1: 'Malignant'})
ct[['Benign', 'Malignant']].plot.bar(stacked=True, ax=axes[0],
                                      color=['#58D68D', '#E74C3C'], edgecolor='white', width=0.8)
axes[0].set_ylim(0, 1); axes[0].set_ylabel('proportion'); axes[0].set_xlabel('age band')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_title('Malignancy rate by age band', fontsize=9.5, fontweight='bold')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8.5, frameon=False)
# Right: most malignant-prone anatomical regions
pd.crosstab(df['region'], df['is_malignant'], normalize='index')[1].sort_values().tail(8).plot.barh(
    ax=axes[1], color='#E74C3C')
axes[1].set_title('Most malignant-prone regions', fontsize=9.5, fontweight='bold')
axes[1].set_xlabel('malignant rate'); axes[1].set_ylabel('region')
fig.suptitle('Attribute relationships with malignancy', fontsize=10, fontweight='bold', y=1.03)
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig7_attributes.png', bbox_inches='tight'); plt.show()
print('Malignancy rate by age band:'); print(df.groupby(ageb)['is_malignant'].mean().round(3).to_string())

In [ ]:
# Geographic variability of malignancy. province/city/postal_code/hospital_id are SYNTHETIC
# (useful for the privacy analysis) -> expect negligible association; report effect size, not only p.
# (hospital_id is nested within province, so we plot province only and report the rest in text.)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
df.groupby('province')['is_malignant'].mean().sort_values().plot.barh(ax=ax, color='#5DADE2')
ax.axvline(df['is_malignant'].mean(), color='#E74C3C', ls='--', lw=1)
ax.set_title('Malignant rate by province (dashed = overall rate)', fontsize=9.5, fontweight='bold')
ax.set_xlabel('malignant rate')
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_geo.png', bbox_inches='tight'); plt.show()
print("Geography vs malignancy (Cramer's V = effect size; p from chi-square):")
for c in ['province', 'city', 'hospital_id']:
    v, p = cramers_v(df[c].astype(str), df['is_malignant'].astype(str))
    print(f'  {c:12s} V={v:.3f}  p={p:.3f}  -> {"negligible effect (synthetic geo)" if v < 0.1 else "non-trivial"}')

## Step 5 - Evaluation and improvement proposals

In [ ]:
# Step 5 demo: score three candidate dataset versions on six data-quality dimensions in [0,1].
# CLEANED maps booleans to {0,1}, encodes the MNAR block as not_assessed (+ a clinical_assessed
# indicator) and drops identifiers; REDUCED also drops the redundant diameter_2.
BOOL = ['smoke','drink','pesticide','skin_cancer_history','cancer_history','has_piped_water',
        'has_sewage_system','itch','grew','hurt','changed','bleed','elevation','biopsed']
CATNA = ['background_father','background_mother']; NUM_MNAR = ['fitspatrick','diameter_1','diameter_2']
IDS = ['dni','patient_id','lesion_id','img_id']; VALID_DX = {'BCC','MEL','SCC','ACK','NEV','SEK','BOD'}
def to01(s): return s.astype(str).str.strip().str.upper().map({'TRUE':1,'FALSE':0,'YES':1,'NO':0})

raw = df.copy()
cleaned = df.copy()
for c in BOOL:
    m = to01(cleaned[c]); cleaned[c] = m.where(m.notna(), 'not_assessed')   # UNK/NaN -> explicit category
for c in CATNA: cleaned[c] = cleaned[c].where(cleaned[c].notna(), 'not_assessed')
cleaned['clinical_assessed'] = (~df[NUM_MNAR].isna().all(axis=1)).astype(int)  # indicator for numeric block
cleaned = cleaned.drop(columns=IDS)
reduced = cleaned.drop(columns=['diameter_2'])   # redundant with diameter_1 (r=0.89)

def quality_dimensions(d):
    n = len(d); feat = list(d.columns)
    completeness = 1 - d[feat].isna().sum().sum() / (n*len(feat))
    uniqueness   = 1 - d.duplicated().sum() / n
    ch = []
    if 'age' in d: ch.append(d['age'].between(0,120) | d['age'].isna())
    if 'fitspatrick' in d: ch.append(d['fitspatrick'].isin([1,2,3,4,5,6]) | d['fitspatrick'].isna())
    if 'diagnostic' in d: ch.append(d['diagnostic'].isin(VALID_DX))
    for c in [x for x in BOOL if x in d]: ch.append(d[c].isin([0,1,'not_assessed']) | d[c].isna())
    validity = float(pd.concat(ch, axis=1).values.mean())
    bl = [x for x in BOOL if x in d]
    cons_bool = float(d[bl].apply(lambda col: col.map(lambda v: (v in (0,1,0.0,1.0,'not_assessed')) or pd.isna(v))).values.mean()) if bl else 1.0
    if 'diameter_1' in d and 'diameter_2' in d:
        b = d[['diameter_1','diameter_2']].dropna(); cons_d = float((b['diameter_1']>=b['diameter_2']).mean()) if len(b) else 1.0
    else: cons_d = 1.0
    acc = []
    if 'age' in d: acc.append(d['age'].between(0,110) | d['age'].isna())
    if 'diameter_1' in d: acc.append((d['diameter_1']>=0) | d['diameter_1'].isna())
    accuracy = float(pd.concat(acc, axis=1).values.mean()) if acc else 1.0
    integrity = 1 - sum(c in d.columns for c in IDS) / len(IDS)
    return {'completeness': round(completeness,3), 'uniqueness': round(uniqueness,3),
            'validity': round(validity,3), 'consistency': round((cons_bool+cons_d)/2,3),
            'accuracy': round(accuracy,3), 'integrity': round(integrity,3)}
QUAL = pd.DataFrame({'RAW': quality_dimensions(raw), 'CLEANED': quality_dimensions(cleaned),
                     'REDUCED': quality_dimensions(reduced)}).T
print('Dataset versions ->', {'RAW': raw.shape, 'CLEANED': cleaned.shape, 'REDUCED': reduced.shape})
print(QUAL.to_string())
print('Encoding the MNAR block as not_assessed raises completeness from ~0.85 to ~0.95 with no row loss.')
print('Uniqueness drops because records share quasi-identifier profiles once identifiers are removed.')

In [ ]:
# Quantify the data-leakage risk of a naive image-level split (Step 5 evidence)
idx_all = np.arange(len(df))
tr, te = train_test_split(idx_all, test_size=0.2, random_state=RANDOM_STATE, stratify=df[TARGET])
leak = set(df.iloc[tr]['patient_id']) & set(df.iloc[te]['patient_id'])
print(f'Image-level split: {len(leak)} patients appear in BOTH train and test '
      f'({len(leak)/df["patient_id"].nunique()*100:.1f}% of patients leak)')
g1, g2 = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
              .split(idx_all, groups=df['patient_id']))
print(f'Patient-grouped split: {len(set(df.iloc[g1]["patient_id"]) & set(df.iloc[g2]["patient_id"]))} patients leak')